# Coloration black-box — the SOTA LSTM on the amplitude-matched signal

**Google Colab**: Runtime → **GPU**. Same Drive/dataset/split plumbing as
`train_lstm_gain_prior_ws.ipynb`. **Push local changes before running.**

## Idea — obedient gain + proven coloration trunk, no tricks

The 07_experiments/08 GR-source sweep splits the model zoo cleanly:
the gain-prior family **obeys** the GR input best (const_0db: output pinned to
the commanded gain) but never learns the coloration (waveshaper → identity,
metrics parked at amp-match); GR-TFiLM **learns** the coloration to near-SOTA
ESR (oracle 0.0008 A-wt) but half-ignores the GR input. This model takes the
obedient multiply and hands ONLY the coloration to the unchanged from-scratch
SOTA trunk:

```
matched = x · 10^(gr/20)                    ← gain: explicit multiply, obeys
y = tanh( lin( LSTM(matched ⊕ tvcond) ) )   ← coloration: SOTA LSTM32TVC trunk
```

- **The only difference from the 02b SOTA black-box is the input signal**
  (matched instead of dry). The trunk provably learns saturation + filtering
  at this size from raw dry (≈0.0025 A-wt ESR); here it only has to learn the
  easier matched → wet mapping (~18 % RMS residual).
- **It never sees the GR value**, so there is nothing to disobey — the
  gain-following of the 08 sweep is inherited by construction (sidechain goal).
- **No zero-init anchor, no Δg, no waveshaper, no extra heads.** The untrained
  model is NOT the amplitude match; cell 6 prints the amp-match L1 as the
  baseline the run must beat.
- Trained on the **oracle** GR curves (this notebook, 06_output lineage):
  training on predicted GR is what taught the e2e cascade to distrust its GR
  input. Predicted/const GR behaviour is evaluated by inference-time swap in
  `07_experiments/08_gr_source_ablation_multimodel.ipynb`.

## Model — `model_color_lstm.ColorBlackboxLSTM`, 8,033 params
Parameter-matched to SOTA LSTM32TVC (8k) and the gain-prior variants (8.3–8.4k).
Same v2 seed-42 split, 3 s crops, TBPTT 4410, AdamW + cosine, 100 epochs, bf16,
4c loss. Reuses `GainPriorSystem` + `GRCropDataModule` unchanged
(`return_parts` reports Δg≡0 and color = y − matched, so the `gain/*` logs read
"total learned coloration").

Comparable runs: from-scratch SOTA LSTM32TVC (same trunk, dry input), oracle
`gain_prior_ws`, `gr_tfilm`, and the raw amplitude match.


In [ ]:
# -- 0. Dependencies ---------------------------------------------------
# Uses nablafx (TVFiLMCond). Pin numpy first so lightning/nablafx installs
# can't downgrade Colab's numpy 2.x and break torch. Install lightning/nablafx
# --no-deps so they can't clobber Colab's CUDA torch. `rational` /
# `frechet_audio_distance` are nablafx import-chain deps we never use; stub
# both so `from nablafx...` doesn't drag in broken wheels.
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss einops lightning-utilities packaging
!pip install -q --no-deps lightning nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

fad = types.ModuleType("frechet_audio_distance")
fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules["frechet_audio_distance"] = fad

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} - restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")

In [ ]:
# -- 1. Mount Drive (dataset) + clone repo from GitHub (code) ---------
# The repo is NOT synced to Drive (only data/ is). Code comes from GitHub -
# push local changes before (re)running this cell; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" reset --hard origin/main
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT

# Module directory for this notebook (dataset_tfilm / model_gainprior_ws / ... live here).
COND_DIR = os.path.join(REPO_ROOT, "06_output")
assert os.path.isfile(os.path.join(COND_DIR, "model_gainprior_ws.py")), (
    f"Clone failed or stale: {COND_DIR}. Did you push local changes?"
)

OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "diffssl_gain_prior_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), f"Bad DATA_ROOT: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, "processed_ground_truth")), "Missing wet audio dir"
assert os.path.isdir(os.path.join(DATA_ROOT, "test_ground_truth")), (
    f"No test_ground_truth/ under {DATA_ROOT} — it defines the held-out test set; "
    "sync it to Drive before training."
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drop cached local modules so a prior run cannot keep stale classes.
for _name in list(sys.modules):
    if _name in ("dataset_tfilm", "model_tfilm", "model_gainprior",
                 "model_gainprior_ws", "system_gainprior", "splits",
                 "amplitude_match"):
        del sys.modules[_name]

# repo root (for `src` + `nablafx`) + module dir
for p in (REPO_ROOT, os.path.join(REPO_ROOT, "nablafx"), COND_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"COND_DIR   : {COND_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

In [ ]:
# -- 2. Cache dataset to Colab local SSD ------------------------------
# Same cache as the additive gain-prior notebook — dry WAV per song, plus
# gr_curve (.pt) + wet WAV per (song, setting) pair — with wet/gr mirrored
# under their ORIGINAL subfolders, so the external test pairs stay under
# test_ground_truth/ on the local cache too (the split treats that folder
# as test-only).

import shutil
from dataset_tfilm import discover_gr_pairs

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"

pairs = discover_gr_pairs(DATA_ROOT, include_test_ground_truth=True)
settings = sorted({p["setting"] for p in pairs})
songs = sorted({p["song"] for p in pairs})
print(f"Caching {len(songs)} songs x {len(settings)} settings ({len(pairs)} pairs) -> {LOCAL_DATA_ROOT}")

def _mirror(src, dst):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

# dry WAVs (one per song, shared across settings)
for song in songs:
    fn = f"{song}_UnmasteredWAV.wav"
    _mirror(Path(DATA_ROOT) / "processed_normalized" / fn,
            Path(LOCAL_DATA_ROOT) / "processed_normalized" / fn)

# GR curves (.pt) + wet WAVs (-exported.wav), per (song, setting) pair,
# preserving the source subfolder (processed_ground_truth vs test_ground_truth)
for p in pairs:
    _mirror(p["gr"], Path(LOCAL_DATA_ROOT) / Path(p["gr"]).relative_to(DATA_ROOT))
    _mirror(p["wet"], Path(LOCAL_DATA_ROOT) / Path(p["wet"]).relative_to(DATA_ROOT))

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

In [ ]:
# -- 3. Imports & hyper-parameters (coloration black-box + 4c loss) --

import importlib
import json
from datetime import datetime

import torch
import lightning as pl
from lightning.pytorch.callbacks import (
    LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

import dataset_tfilm as _dataset_tfilm
importlib.reload(_dataset_tfilm)
from dataset_tfilm import (
    BATCH_SIZE, SAMPLE_LENGTH, SAMPLE_RATE, GRCropDataModule, discover_gr_pairs,
)

import model_color_lstm as _model_color_lstm
importlib.reload(_model_color_lstm)
from model_color_lstm import ColorBlackboxLSTM

import system_gainprior as _system_gainprior   # reused UNCHANGED from the additive nb
importlib.reload(_system_gainprior)
from system_gainprior import GainPriorSystem

from splits import (
    DIFFSSL_PARAM_RANGES, build_split_manifest, discover_test_ground_truth_keys,
)
from src.dsp import PARAM_ORDER

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split: v2 external test_ground_truth policy (test pairs excluded by key) --
SPLIT_SEED   = 42
N_VAL_SONGS  = 1

# -- training (same diffssl TBPTT regime as the additive gain-prior nb) --
LR               = 1e-3
MAX_EPOCHS       = 100      # fixed budget == cosine T_max
STEP_NUM_SAMPLES = 4410     # diffssl TBPTT sub-step (0.1 s)
SCHEDULER        = "cosine"
ETA_MIN          = 1e-6
USE_AMP          = True
CHECK_VAL_EVERY_N_EPOCH = 1

# -- model core (identical to the additive gain-prior run) --
HIDDEN_SIZE     = 32
NUM_LAYERS      = 1
NUM_CONTROLS    = 4
TVCOND_DIM      = 16
COND_BLOCK_SIZE = 128
COND_NUM_LAYERS = 1
# -- loss (4c, unchanged). ENV_WEIGHT=0, PE_WEIGHT=0, MRSTFT_VARIANT="sota" == exact 02b --
TD_WEIGHT      = 0.5      # L1
FD_WEIGHT      = 0.5      # MR-STFT
ENV_WEIGHT     = 0.2      # RMS-envelope L1 in dB (differentiable GR MAE, window 1024)
PE_WEIGHT      = 0.1      # pre-emphasis L1 (transients)
MRSTFT_VARIANT = "extended"   # "extended" (adds 4096-FFT + lin-mag) | "sota"

RUN_TAG    = "diffssl_lstm32_color_bb"
RESUME_RUN = None

In [ ]:
# -- 4. Preview split — v2 external test_ground_truth policy ----------
# The ONLY held-out test set is test_ground_truth/ (scanned on DRIVE; the
# local cache mirrors it under the same subfolder). Its pairs are excluded
# from train/val BY KEY; every remaining (song, setting) pair trains or
# validates — including Air/NosPalpitants and the lowest-threshold settings.

TEST_KEYS = discover_test_ground_truth_keys(DRIVE_DATA_ROOT)
print(f"test_ground_truth pairs ({len(TEST_KEYS)}):")
for k in sorted(TEST_KEYS):
    print(f"  {k}")

preview = build_split_manifest(
    discover_gr_pairs(DATA_ROOT, include_test_ground_truth=True),
    seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS,
    test_pair_keys=TEST_KEYS,
)
leaked = (set(preview.train_pair_keys) | set(preview.val_pair_keys)) & TEST_KEYS
assert not leaked, f"test_ground_truth pairs leaked into train/val: {sorted(leaked)}"

print(f"\nSettings ({len(preview.all_settings)}): {preview.all_settings}")
print(f"Train songs: {preview.train_songs}")
print(f"Val songs  : {preview.val_songs}")
print(f"Test pairs : {preview.test_pair_keys}")
print(f"Pairs - train={len(preview.train_pair_keys)} "
      f"val={len(preview.val_pair_keys)} test={len(preview.test_pair_keys)}")

In [ ]:
# -- 5. Model size ----------------------------------------------------

model = ColorBlackboxLSTM(
    num_controls=NUM_CONTROLS, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
    tvcond_dim=TVCOND_DIM, cond_block_size=COND_BLOCK_SIZE, cond_num_layers=COND_NUM_LAYERS,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"ColorBlackboxLSTM: {n_params:,} params  "
      f"(hidden={HIDDEN_SIZE}, tvcond_dim={TVCOND_DIM}, controls={NUM_CONTROLS})")
for name, mod in model.named_children():
    print(f"  {name:10s} {sum(p.numel() for p in mod.parameters()):,}")
print(f"\nSOTA LSTM32TVC is 8k (same trunk, dry input); gain-prior variants 8.3-8.4k "
      f"- parameter-matched.")
print(f"Crop {SAMPLE_LENGTH} ({SAMPLE_LENGTH/SAMPLE_RATE:.2f}s) | {SAMPLE_RATE} Hz | "
      f"TBPTT step {STEP_NUM_SAMPLES} | tvcond block {COND_BLOCK_SIZE}")


In [ ]:
# -- 6. DataModule + amp-match baseline --------------------------------
# NO zero-init anchor here (that's the point): the untrained model is a
# randomly-initialised readout, NOT the amplitude match. Print both L1s on a
# real batch — the amp-match number is the baseline training must beat.

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

NUM_WORKERS = min(8, os.cpu_count() or 2)
print(f"DataLoader num_workers: {NUM_WORKERS}")

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"color_bb_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = GRCropDataModule(
    data_root=DATA_ROOT, sample_length=SAMPLE_LENGTH, sample_rate=SAMPLE_RATE,
    batch_size=BATCH_SIZE, split_seed=SPLIT_SEED, n_val_songs=N_VAL_SONGS,
    test_gt_root=DRIVE_DATA_ROOT,   # test keys scanned on Drive (source of truth)
    split_manifest_path=split_path, num_workers=NUM_WORKERS,
)
dm.setup()
print(f"Train/val/test crops: {len(dm.train_dataset)} / {len(dm.val_dataset)} / {len(dm.test_dataset)}")
print(f"Batches/epoch (train): {len(dm.train_dataloader())}  (batch_size={BATCH_SIZE})")

# -- baselines on a real val batch --------------------------------------
from amplitude_match import amplitude_match

if not RESUME_RUN:
    _dry, _gr, _wet, _p = next(iter(dm.val_dataloader()))
    with torch.no_grad():
        model.reset_states()
        _y0 = model(_dry, _gr, _p)
    _l1_untrained = float(torch.nn.functional.l1_loss(_y0, _wet))
    _l1_amp = float(torch.nn.functional.l1_loss(amplitude_match(_dry, _gr), _wet))
    print(f"untrained model crop L1 vs wet : {_l1_untrained:.6f}  (random readout)")
    print(f"amp-match      crop L1 vs wet : {_l1_amp:.6f}  <- the baseline to beat")
    model.reset_states()
    del _dry, _gr, _wet, _p, _y0

In [ ]:
# -- 7. Train ---------------------------------------------------------

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "color_bb: y = tanh(lin(LSTM(x * 10^(gr/20), tvcond))) - SOTA trunk "
                    "on the amplitude-matched signal; gain by explicit multiply",
        "model_type": "ColorBlackboxLSTM",
        "model_ref": "SOTA LSTM32TVC trunk (dry->matched input swap); motivated by the 08 "
                     "GR-source sweep (gain-prior obeys / tfilm colors) + simplify ablation",
        "dataset": "Diff-SSL-G-Comp", "setting": "multi (all non-test settings, tvcond on 4 knobs)",
        "conditioning": "knobs via tvcond (TVFiLMCond); GR only via the input multiply "
                        "(model never sees the GR value)",
        "sample_rate": SAMPLE_RATE, "sample_length": SAMPLE_LENGTH, "batch_size": BATCH_SIZE,
        "step_num_samples": STEP_NUM_SAMPLES,
        "param_order": PARAM_ORDER, "param_ranges": DIFFSSL_PARAM_RANGES,
        "split_seed": SPLIT_SEED,
        "split_policy": "external_test_ground_truth (test pairs excluded from train/val by key)",
        "test_pair_keys": sorted(TEST_KEYS),
        "train_songs": dm.split.train_songs,
        "val_songs": dm.split.val_songs, "test_songs": dm.split.test_songs,
        "test_settings": dm.split.test_settings,
        "model": {"hidden_size": HIDDEN_SIZE, "num_layers": NUM_LAYERS,
                   "num_controls": NUM_CONTROLS, "tvcond_dim": TVCOND_DIM,
                   "cond_block_size": COND_BLOCK_SIZE, "cond_num_layers": COND_NUM_LAYERS,
                   "num_params": n_params},
        "loss": {"td_weight": TD_WEIGHT, "fd_weight": FD_WEIGHT,
                 "env_weight": ENV_WEIGHT, "pe_weight": PE_WEIGHT,
                 "mrstft_variant": MRSTFT_VARIANT,
                 "kind": "td*L1 + fd*MRSTFT + env*envdB_L1 + pe*preemph_L1"},
        "metrics": ["esr", "rmse", "mae", "mse"],
        "optimizer": f"adamw + {SCHEDULER}",
        "scheduler": SCHEDULER, "eta_min": ETA_MIN, "use_amp": USE_AMP,
        "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
        "training": "diffssl_crop_batches + tbptt_substeps (reset each batch)",
        "lr": LR, "max_epochs": MAX_EPOCHS,
    }, f, indent=2)

system = GainPriorSystem(
    model=model, lr=LR, step_num_samples=STEP_NUM_SAMPLES,
    td_weight=TD_WEIGHT, fd_weight=FD_WEIGHT,
    env_weight=ENV_WEIGHT, pe_weight=PE_WEIGHT, mrstft_variant=MRSTFT_VARIANT,
    scheduler=SCHEDULER, max_epochs=MAX_EPOCHS, eta_min=ETA_MIN, use_amp=USE_AMP,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(dirpath=ckpt_dir, monitor="loss/val", mode="min", save_top_k=3,
                    save_last=True, filename="best-{epoch:03d}-{step}",
                    auto_insert_metric_name=False),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
    callbacks=callbacks, logger=loggers, log_every_n_steps=10,
    check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
)
trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")

In [ ]:
# -- 8. Test (external test_ground_truth pairs — unseen songs & settings) --

best_ckpt = callbacks[0].best_model_path or os.path.join(ckpt_dir, "last.ckpt")
print(f"Testing with: {best_ckpt}")
trainer.test(system, datamodule=dm, ckpt_path=best_ckpt)

In [ ]:
# -- 9. Plot: prediction vs target + coloration share -------------------
# For each example: (top) waveform overlay, (bottom) the GR input driving the
# multiply. mean|color| = mean |y - matched|, the total learned coloration.

import matplotlib.pyplot as plt
import numpy as np
from system_gainprior import esr_metric

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

val_batches = list(dm.val_dataloader())
dry, gr, wet, params = val_batches[len(val_batches) // 2]

with torch.no_grad():
    system.model.reset_states()
    pred, delta_db, color, ws_res = system.model(
        dry.cuda(), gr.cuda(), params.cuda(), return_parts_full=True)
    pred, delta_db = pred.cpu(), delta_db.cpu()
    color, ws_res = color.cpu(), ws_res.cpu()

dry_np, wet_np, pred_np = dry.numpy(), wet.numpy(), pred.numpy()
gr_np, delta_np = gr.numpy(), delta_db.numpy()
color_np, ws_np = color.numpy(), ws_res.numpy()
n_plots = min(3, dry_np.shape[0])
fig, axes = plt.subplots(2 * n_plots, 1, figsize=(14, 4.6 * n_plots), squeeze=False)
t = np.arange(wet_np.shape[-1]) / SAMPLE_RATE
for r in range(n_plots):
    ax = axes[2 * r, 0]
    ax.plot(t, dry_np[r, 0], label="Dry", alpha=0.35, lw=0.5, color="gray")
    ax.plot(t, wet_np[r, 0], label="Target (wet)", alpha=0.8, lw=0.5)
    ax.plot(t, pred_np[r, 0], label="Predicted", alpha=0.8, lw=0.5)
    pv = torch.from_numpy(pred_np[r]); tv = torch.from_numpy(wet_np[r])
    mae_r = float(np.mean(np.abs(pred_np[r, 0] - wet_np[r, 0])))
    ax.set_title(f"crop {r} - MAE {mae_r:.4f} | ESR {float(esr_metric(tv, pv)):.4f} | "
                 f"mean|color| {np.abs(color_np[r]).mean():.5f}")
    ax.set_ylabel("amp"); ax.legend(loc="lower right", fontsize=8); ax.set_ylim(-1.05, 1.05)

    ax = axes[2 * r + 1, 0]
    ax.plot(t, gr_np[r, 0], label="GR input (dB) - applied verbatim", lw=0.7,
            color="#1f77b4")
    ax.set_ylabel("gain (dB)"); ax.legend(loc="lower right", fontsize=8)
axes[-1, 0].set_xlabel("Time (s)")
fig.suptitle(f"Coloration black-box LSTM - best val loss {callbacks[0].best_model_score:.6f}", y=1.002)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_output_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()

In [ ]:
# -- 10. Learned coloration spectra ------------------------------------
# The reason this model exists: is the matched->wet residual actually being
# synthesised now? Compare PSDs of the TARGET coloration (wet - matched) and
# the LEARNED coloration (pred - matched) on the plotted val batch. Overlap
# above ~300 Hz (where the gain-prior runs left the residual untouched) is
# the win condition.

from scipy.signal import welch

matched_v = (dry * torch.pow(10.0, gr.clamp(-30.0, 5.0) / 20.0)).numpy()
tgt_col = (wet_np - matched_v)[:, 0].reshape(-1)
lrn_col = (pred_np - matched_v)[:, 0].reshape(-1)
wet_flat = wet_np[:, 0].reshape(-1)

fig, ax = plt.subplots(figsize=(11, 4.5))
for sig, name, style in ((wet_flat, "wet", dict(color="k", lw=1.2)),
                         (tgt_col, "target coloration (wet - matched)",
                          dict(color="#1f77b4", lw=1.2)),
                         (lrn_col, "learned coloration (pred - matched)",
                          dict(color="#d62728", lw=1.0))):
    f, Pxx = welch(sig, fs=SAMPLE_RATE, nperseg=8192)
    ax.semilogx(f, 10 * np.log10(Pxx + 1e-18), label=name, **style)
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("PSD (dB/Hz)")
rms = lambda v: float(np.sqrt(np.mean(v ** 2)))
ax.set_title(f"Coloration spectra - target {rms(tgt_col)/rms(wet_flat):.1%} of wet RMS, "
             f"learned {rms(lrn_col)/rms(wet_flat):.1%}")
ax.grid(alpha=0.3, which="both"); ax.legend(); ax.set_xlim(20, SAMPLE_RATE / 2)
fig.tight_layout()
spec_path = os.path.join(RUN_DIR, "eval_coloration_spectra.png")
fig.savefig(spec_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {spec_path}")
plt.show()


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"